In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from scipy.stats import rankdata, norm
import warnings
warnings.filterwarnings('ignore')

import gstools as gs


In [2]:
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120

In [4]:
url = "/home/wasif/sds/data/2D_MV_200Wells.csv"

# Read the data
mydata = pd.read_csv(url)

# Display first few rows
print("First 6 rows of the dataset:")
print(mydata.head())

print("\nColumn names:", mydata.columns.tolist())
print(f"Dataset shape: {mydata.shape}")

First 6 rows of the dataset:
      X     Y  facies_threshold_0.3  porosity  permeability  \
0   565  1485                     1    0.1184         6.170   
1  2585  1185                     1    0.1566         6.275   
2  2065  2865                     2    0.1920        92.297   
3  3575  2655                     1    0.1621         9.048   
4  1835    35                     1    0.1766         7.123   

   acoustic_impedance  
0               2.009  
1               2.864  
2               3.524  
3               2.157  
4               3.979  

Column names: ['X', 'Y', 'facies_threshold_0.3', 'porosity', 'permeability', 'acoustic_impedance']
Dataset shape: (200, 6)


## Part 1: Non-Normalized Isotropic Variogram Calculation

### Understanding Non-Normalized Variograms

**Key Concept**: A variogram measures spatial dependence without transforming the data. We work directly with raw values.

The experimental semivariogram is calculated as:

$$\gamma(h) = \frac{1}{2N(h)} \sum_{i=1}^{N(h)} [Z(x_i) - Z(x_i + h)]^2$$

Where:
- $h$ = lag distance
- $N(h)$ = number of pairs at distance $h$
- $Z(x_i)$ = raw value at location $x_i$
- $Z(x_i + h)$ = value at distance $h$ away

**Advantage over normalized**: Variograms preserve the original data variance, making results directly interpretable in original units.

In [ ]:
from scipy.optimize import curve_fit

# Extract coordinates and raw porosity values (NO normalization)
coords = mydata[['X', 'Y']].values.T  # Shape: (2, 200)
values_raw = mydata['porosity'].values  # Raw porosity values

print("=" * 70)
print("ISOTROPIC VARIOGRAM - NON-NORMALIZED (Raw Porosity)")
print("=" * 70)

# Calculate statistics for reference
print(f"\nOriginal Data Statistics:")
print(f"  Mean porosity: {values_raw.mean():.4f}")
print(f"  Std dev: {values_raw.std():.4f}")
print(f"  Variance: {values_raw.var():.4f}")
print(f"  Min: {values_raw.min():.4f}, Max: {values_raw.max():.4f}")

# Define binning parameters
max_dist = 3000  # Maximum lag distance (m)
bin_width = 400  # Lag class width (m)
n_bins = int(max_dist / bin_width)
bin_edges = np.linspace(0, max_dist, n_bins + 1)

print(f"\nVariogram Calculation Settings:")
print(f"  Max lag distance: {max_dist} m")
print(f"  Bin width: {bin_width} m")
print(f"  Number of bins: {n_bins}")

# Calculate omnidirectional (isotropic) variogram using gstools
lags_iso, gamma_iso = gs.vario_estimate(
    coords,              # Transposed coordinates (2, n_points)
    values_raw,          # Raw porosity values
    bin_edges=bin_edges,
    direction=None,      # None = omnidirectional
    return_counts=False
)

print(f"\nIsotropic Variogram Results:")
print(f"  Computed {len(lags_iso)} lag classes")
print(f"  Lag centers (first 5): {lags_iso[:5]}")
print(f"  Gamma values (first 5): {gamma_iso[:5]:.4f}")

# Plot isotropic variogram
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(lags_iso, gamma_iso, c='darkblue', s=80, zorder=5, 
           label='Experimental variogram (raw porosity)', edgecolors='black', linewidth=1.5)

ax.set_xlabel('Lag Distance h (m)', fontsize=12, fontweight='bold')
ax.set_ylabel('Semivariogram $\gamma(h)$', fontsize=12, fontweight='bold')
ax.set_title('Isotropic Variogram - Non-Normalized Porosity', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('isotropic_variogram_raw.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Isotropic variogram calculated and plotted")

### Fitting Theoretical Variogram Models

We fit three common models to the experimental variogram to characterize spatial structure:

**1. Spherical Model**: 
$$\gamma(h) = \begin{cases} c_0 + c \left[\frac{3h}{2a} - \frac{1}{2}\left(\frac{h}{a}\right)^3\right] & h \leq a \\ c_0 + c & h > a \end{cases}$$

**2. Exponential Model**: 
$$\gamma(h) = c_0 + c \left[1 - \exp\left(-\frac{3h}{a}\right)\right]$$

**3. Gaussian Model**: 
$$\gamma(h) = c_0 + c \left[1 - \exp\left(-3\left(\frac{h}{a}\right)^2\right)\right]$$

Where:
- $c_0$ = **nugget** (variance at zero distance, e.g., measurement error)
- $c$ = **sill contribution** (partial sill)
- $c_0 + c$ = **total sill** (should equal data variance for well-behaved data)
- $a$ = **range** (distance at which spatial correlation becomes negligible)

In [ ]:
def spherical_model(h, nugget, sill, range_):
    """
    Spherical variogram model
    Parameters: nugget, partial sill, range
    """
    gamma = np.zeros_like(h, dtype=float)
    mask = h <= range_
    gamma[mask] = nugget + sill * (1.5 * (h[mask]/range_) - 0.5 * (h[mask]/range_)**3)
    gamma[~mask] = nugget + sill
    return gamma

def exponential_model(h, nugget, sill, range_):
    """
    Exponential variogram model
    Effective range ≈ 3 × range parameter
    """
    return nugget + sill * (1 - np.exp(-3 * h / range_))

def gaussian_model(h, nugget, sill, range_):
    """
    Gaussian variogram model
    Effective range ≈ sqrt(3) × range parameter
    """
    return nugget + sill * (1 - np.exp(-3 * (h / range_)**2))

# Fit all three models to the isotropic variogram
# Use lag classes away from zero to avoid fitting noise at small distances
valid_mask = (lags_iso > 200) & (lags_iso < 2500)

print("\n" + "=" * 70)
print("FITTING THEORETICAL MODELS TO ISOTROPIC VARIOGRAM")
print("=" * 70)

# Get data statistics for bounds
total_variance = values_raw.var()
print(f"\nReference: Total data variance = {total_variance:.4f}")
print(f"Using lags between 200-2500 m for fitting ({np.sum(valid_mask)} points)\n")

# 1. FIT SPHERICAL MODEL
try:
    params_sph, _ = curve_fit(
        spherical_model,
        lags_iso[valid_mask],
        gamma_iso[valid_mask],
        p0=[0.5, total_variance * 0.5, 800],
        bounds=([0, 0, 100], [total_variance, total_variance, 3000]),
        maxfev=5000
    )
    nugget_sph, sill_sph, range_sph = params_sph
    
    print(f"Spherical Model:")
    print(f"  Nugget (c₀): {nugget_sph:.4f}")
    print(f"  Partial Sill (c): {sill_sph:.4f}")
    print(f"  Total Sill (c₀ + c): {nugget_sph + sill_sph:.4f}")
    print(f"  Range (a): {range_sph:.1f} m")
    print(f"  ✓ Fitted successfully\n")
except Exception as e:
    print(f"✗ Spherical fit failed: {e}\n")
    params_sph = None

# 2. FIT EXPONENTIAL MODEL
try:
    params_exp, _ = curve_fit(
        exponential_model,
        lags_iso[valid_mask],
        gamma_iso[valid_mask],
        p0=[0.5, total_variance * 0.5, 600],
        bounds=([0, 0, 100], [total_variance, total_variance, 3000]),
        maxfev=5000
    )
    nugget_exp, sill_exp, range_exp = params_exp
    
    print(f"Exponential Model:")
    print(f"  Nugget (c₀): {nugget_exp:.4f}")
    print(f"  Partial Sill (c): {sill_exp:.4f}")
    print(f"  Total Sill (c₀ + c): {nugget_exp + sill_exp:.4f}")
    print(f"  Range parameter (a): {range_exp:.1f} m")
    print(f"  Effective range (≈3a): {range_exp * 3:.1f} m")
    print(f"  ✓ Fitted successfully\n")
except Exception as e:
    print(f"✗ Exponential fit failed: {e}\n")
    params_exp = None

# 3. FIT GAUSSIAN MODEL
try:
    params_gau, _ = curve_fit(
        gaussian_model,
        lags_iso[valid_mask],
        gamma_iso[valid_mask],
        p0=[0.5, total_variance * 0.5, 700],
        bounds=([0, 0, 100], [total_variance, total_variance, 3000]),
        maxfev=5000
    )
    nugget_gau, sill_gau, range_gau = params_gau
    
    print(f"Gaussian Model:")
    print(f"  Nugget (c₀): {nugget_gau:.4f}")
    print(f"  Partial Sill (c): {sill_gau:.4f}")
    print(f"  Total Sill (c₀ + c): {nugget_gau + sill_gau:.4f}")
    print(f"  Range parameter (a): {range_gau:.1f} m")
    print(f"  Effective range (≈√3·a): {range_gau * np.sqrt(3):.1f} m")
    print(f"  ✓ Fitted successfully\n")
except Exception as e:
    print(f"✗ Gaussian fit failed: {e}\n")
    params_gau = None

# Plot all fitted models
h_smooth = np.linspace(0, 3000, 300)

fig, ax = plt.subplots(figsize=(12, 7))

# Experimental variogram
ax.scatter(lags_iso, gamma_iso, c='black', s=100, zorder=5, 
           label='Experimental variogram', edgecolors='black', linewidth=2)

# Plot fitted models
if params_sph is not None:
    gamma_sph = spherical_model(h_smooth, *params_sph)
    ax.plot(h_smooth, gamma_sph, 'b-', linewidth=2.5, 
            label=f'Spherical (nugget={nugget_sph:.2f}, range={range_sph:.0f}m)')

if params_exp is not None:
    gamma_exp = exponential_model(h_smooth, *params_exp)
    ax.plot(h_smooth, gamma_exp, 'r--', linewidth=2.5, 
            label=f'Exponential (nugget={nugget_exp:.2f}, range={range_exp:.0f}m)')

if params_gau is not None:
    gamma_gau = gaussian_model(h_smooth, *params_gau)
    ax.plot(h_smooth, gamma_gau, 'g:', linewidth=2.5, 
            label=f'Gaussian (nugget={nugget_gau:.2f}, range={range_gau:.0f}m)')

# Reference line for total sill
ax.axhline(y=total_variance, color='orange', linestyle='--', alpha=0.6, 
           linewidth=2, label=f'Total Sill (variance={total_variance:.2f})')

ax.set_xlabel('Lag Distance h (m)', fontsize=12, fontweight='bold')
ax.set_ylabel('Semivariogram $\gamma(h)$', fontsize=12, fontweight='bold')
ax.set_title('Isotropic Variogram: Non-Normalized Porosity with Model Fits', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 3000)

plt.tight_layout()
plt.savefig('isotropic_variogram_models_raw.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Model comparison plot saved")